In [ ]:
import pandas as pd
import ipywidgets as widgets
from ipywidgets import interactive
import matplotlib.pyplot as plt
from IPython.display import clear_output

In [ ]:
%matplotlib inline

In [ ]:
# Data for the database
database = [
    {"Fever": 1, "Cough": 1, "Low_Oxygen": 1, "COVID": 1},
    {"Fever": 0, "Cough": 1, "Low_Oxygen": 0, "COVID": 0},
    {"Fever": 1, "Cough": 0, "Low_Oxygen": 0, "COVID": 1},
    {"Fever": 1, "Cough": 1, "Low_Oxygen": 0, "COVID": 1},
    {"Fever": 0, "Cough": 0, "Low_Oxygen": 0, "COVID": 0},
    {"Fever": 1, "Cough": 1, "Low_Oxygen": 1, "COVID": 1},
    {"Fever": 0, "Cough": 1, "Low_Oxygen": 1, "COVID": 1},
    {"Fever": 1, "Cough": 0, "Low_Oxygen": 1, "COVID": 0},
    {"Fever": 0, "Cough": 0, "Low_Oxygen": 1, "COVID": 0},
    {"Fever": 0, "Cough": 1, "Low_Oxygen": 0, "COVID": 1},
    {"Fever": 0, "Cough": 1, "Low_Oxygen": 1, "COVID": 1},
    {"Fever": 1, "Cough": 1, "Low_Oxygen": 1, "COVID": 1},
    {"Fever": 0, "Cough": 0, "Low_Oxygen": 1, "COVID": 0},
    {"Fever": 0, "Cough": 0, "Low_Oxygen": 0, "COVID": 0},
    {"Fever": 1, "Cough": 1, "Low_Oxygen": 0, "COVID": 1}
]


In [ ]:
# Convert to DataFrame
df = pd.DataFrame(database)

In [ ]:
# Function to calculate conditional probability
def calcConditionalProbablity(Y_set: pd.DataFrame, X: str, valueX: int):
    counts = Y_set[X].value_counts()
    # Handle the case where valueX might not exist in the subset
    if valueX not in counts:
        return 0
    return counts[valueX] / len(Y_set)


In [ ]:
# Function to predict COVID
def predictCovidYES(sample: dict, df: pd.DataFrame) -> float:
    COVID_YES_SET = df[df["COVID"] == 1]
    COVID_NO_SET = df[df["COVID"] == 0]

    # Calculating P(COVID) and P(¬COVID)
    P_COVID_YES = len(COVID_YES_SET) / len(df)
    P_COVID_NO = len(COVID_NO_SET) / len(df)

    # Calculating P(X1,X2,...Xn|COVID=1)
    P_X_COVID_YES = 1
    for feature in sample.keys():
        P_X_COVID_YES *= calcConditionalProbablity(COVID_YES_SET, feature, sample[feature])

    # Calculating P(X1,X2,...Xn|COVID=0)
    P_X_COVID_NO = 1
    for feature in sample.keys():
        P_X_COVID_NO *= calcConditionalProbablity(COVID_NO_SET, feature, sample[feature])

    # Calculating P(X1,X2,...,Xn)
    P_X_ALL_CLASSES = P_X_COVID_YES * P_COVID_YES + P_X_COVID_NO * P_COVID_NO

    # Avoid division by zero
    if P_X_ALL_CLASSES == 0:
        return 0

    # Bayes Rule
    return (P_X_COVID_YES * P_COVID_YES) / P_X_ALL_CLASSES

In [ ]:
# Function to display prediction
def covid_prediction(fever_status, cough_status, lowO2_status):
    input_sample = {"Fever": fever_status, "Cough": cough_status, "Low_Oxygen": lowO2_status}
    posterior_prob = predictCovidYES(input_sample, df)
    print(f"Posterior P(COVID=1): {round(posterior_prob, 4)*100}%")

    if posterior_prob >= 0.5:
        print("It's Like You Have COVID.(Go to the hospital ASAP!)")
    else:
        print("Chill Dude,(Have a paracetamol and sleep).")

In [ ]:
# Function with plot and output
def covid_prediction_with_plot(fever_status, cough_status, lowO2_status):
    input_sample = {"Fever": fever_status, "Cough": cough_status, "Low_Oxygen": lowO2_status}
    posterior_prob = predictCovidYES(input_sample, df)

    clear_output(wait=True)

    print(f"Posterior P(COVID=1): {round(posterior_prob, 4)*100}%")

    if posterior_prob >= 0.5:
        print("It's Like You Have COVID. (Go to the hospital ASAP!)")
    else:
        print("Chill Dude, (Have a paracetamol and sleep).")

    plt.bar(['COVID Probability'], [posterior_prob], color='tomato')
    plt.ylim(0, 1)
    plt.ylabel('Probability')
    plt.title('Posterior Probability of COVID')
    plt.show()

In [ ]:
# Create widgets for user input
fever_widget = widgets.IntSlider(value=0, min=0, max=1, step=1, description='Fever:')
cough_widget = widgets.IntSlider(value=0, min=0, max=1, step=1, description='Cough:')
lowO2_widget = widgets.IntSlider(value=0, min=0, max=1, step=1, description='Low Oxygen:')

In [ ]:
interactive_plot = interactive(covid_prediction_with_plot, fever_status=fever_widget, cough_status=cough_widget, lowO2_status=lowO2_widget)

#Display Ploting
display(interactive_plot)

interactive(children=(IntSlider(value=0, description='Fever:', max=1), IntSlider(value=0, description='Cough:'…